# k02 — Verified data cache (can-train-and-test v1, DTU DOI 10.11583/DTU.24805533.v1)
CPU only. Downloads the archive once, checks MD5 against the DTU record, unzips to temporary space, checks all 236 CSVs against the Stage 1B SHA-256 digest, and keeps only the verified zip + per-file hashes as notebook output. Experiment notebooks attach this output as read-only input.

In [ ]:
import os, subprocess, hashlib, json, time, shutil
OUT = '/kaggle/working'
ZIP_OUT = OUT + '/can-train-and-test-v1.zip'
URL = 'https://ndownloader.figshare.com/files/43632393'
EXP_MD5 = 'bd6509d670c0a0009cb3ecab34111bcd'
EXP_SIZE = 1507455719
EXP_N, EXP_BYTES = 236, 7471443340
EXP_DIGEST = '15111c4aad65c438c0cce8da77ff904944ab822ac06f3c91163bdd4e5680064b'
rep = {'purpose': 'one-time verified data cache for STAGE2_DESIGN_FROZEN_v1.1 (CPU only)'}
os.makedirs('/kaggle/temp', exist_ok=True)
import glob
t0 = time.time()
prev = [p for p in glob.glob('/kaggle/input/**/can-train-and-test-v1.zip', recursive=True) if os.path.getsize(p) == EXP_SIZE]
rep['source'] = 'attached input (previous version output): ' + prev[0] if prev else 'downloaded from DTU figshare'
if prev:
    shutil.copyfile(prev[0], ZIP_OUT)
for attempt in range(3):
    if os.path.exists(ZIP_OUT) and os.path.getsize(ZIP_OUT) == EXP_SIZE:
        break
    subprocess.run(['wget', '-q', '-c', '-O', ZIP_OUT, URL])
rep['acquire_s'] = round(time.time() - t0, 1)
h = hashlib.md5()
with open(ZIP_OUT, 'rb') as f:
    for b in iter(lambda: f.read(8 << 20), b''): h.update(b)
rep['zip_md5'] = h.hexdigest(); rep['zip_size'] = os.path.getsize(ZIP_OUT)
rep['zip_ok'] = rep['zip_md5'] == EXP_MD5 and rep['zip_size'] == EXP_SIZE
t0 = time.time()
os.makedirs('/kaggle/temp/verify', exist_ok=True)
subprocess.run(['unzip', '-q', '-o', ZIP_OUT, '-d', '/kaggle/temp/verify'], check=True)
ROOT = '/kaggle/temp/verify/can-train-and-test'
paths = sorted(os.path.relpath(os.path.join(dp, fn), ROOT).replace(os.sep, '/') for dp, _, fs in os.walk(ROOT) for fn in fs if fn.endswith('.csv'))
lines = []
for rel in paths:
    p = ROOT + '/' + rel; hh = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(8 << 20), b''): hh.update(b)
    lines.append(rel + ',' + str(os.path.getsize(p)) + ',' + hh.hexdigest())
lines.sort()
digest = hashlib.sha256('\n'.join(lines).encode()).hexdigest()
rep.update(n=len(lines), bytes=sum(int(l.split(',')[1]) for l in lines), digest=digest, verify_s=round(time.time() - t0, 1))
rep['ALL_VERIFIED'] = bool(rep['zip_ok'] and rep['n'] == EXP_N and rep['bytes'] == EXP_BYTES and digest == EXP_DIGEST)
open(OUT + '/file_hashes_sha256.csv', 'w').write('relative_path,size_bytes,sha256\n' + '\n'.join(lines) + '\n')
shutil.rmtree('/kaggle/temp/verify')
json.dump(rep, open(OUT + '/data_cache_report.json', 'w'), indent=1)
print(json.dumps(rep, indent=1))
assert rep['ALL_VERIFIED'], 'VERIFICATION FAILED - do not use this cache'
